# ⚡ Flussi di Lavoro Agente Concorrenti con GitHub Models (Python)

## 📋 Tutorial Avanzato di Elaborazione Parallela

Questo notebook dimostra **modelli di flussi di lavoro concorrenti** utilizzando il Microsoft Agent Framework. Imparerai come costruire flussi di lavoro ad alte prestazioni e con elaborazione parallela dove più agenti AI eseguono simultaneamente, migliorando drasticamente la produttività e consentendo processi aziendali multi-thread sofisticati.

## 🎯 Obiettivi di Apprendimento

### 🚀 **Fondamenti di Elaborazione Concorrente**
- **Esecuzione Parallela degli Agenti**: Eseguire più agenti contemporaneamente per la massima efficienza
- **Orchestrazione dei Flussi di Lavoro**: Coordinare operazioni concorrenti mantenendo la coerenza dei dati
- **Ottimizzazione delle Prestazioni**: Ottenere significativi aumenti di velocità tramite elaborazione parallela
- **Gestione delle Risorse**: Utilizzare in modo efficiente le risorse dei modelli AI attraverso operazioni concorrenti

### 🏗️ **Modelli Avanzati di Concorrenza**
- **Elaborazione Fork-Join**: Suddividere il lavoro tra più agenti e unire i risultati
- **Parallelismo a Pipeline**: Sovrapporre fasi di esecuzione per un throughput continuo
- **Bilanciamento del Carico**: Distribuire il lavoro equamente tra le risorse agente disponibili
- **Punti di Sincronizzazione**: Coordinare agenti concorrenti in fasi critiche del flusso di lavoro

### 🏢 **Applicazioni Aziendali Concorrenti**
- **Elaborazione di Documenti ad Alto Volume**: Elaborare più documenti simultaneamente
- **Analisi Contenuti in Tempo Reale**: Analisi concorrente di flussi di dati in arrivo
- **Ottimizzazione dell’Elaborazione Batch**: Massimizzare il throughput per operazioni su larga scala
- **Analisi Multimodale**: Elaborazione parallela di diversi tipi di contenuti (testo, immagini, dati)

## ⚙️ Prerequisiti e Configurazione

### 📦 **Dipendenze Richieste**

Installa Agent Framework con funzionalità di flussi di lavoro concorrenti:

```bash
pip install agent-framework-core -U
```

### 🔑 **Configurazione GitHub Models**

**Impostazione Ambiente (file .env):**  
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```
  
**Considerazioni sull’Elaborazione Concorrente:**  
- **Limiti di Velocità**: Monitora i limiti API di GitHub Models per richieste concorrenti  
- **Utilizzo delle Risorse**: Considera l’uso di memoria e CPU con più agenti concorrenti  
- **Gestione degli Errori**: Implementa un recupero robusto dagli errori per operazioni parallele

### 🏗️ **Architettura del Flusso di Lavoro Concorrente**

```mermaid
graph TD
    A[Inizio del Flusso di Lavoro] --> B[Esecuzione Concorrente]
    B --> C[Pool di Agenti 1]
    B --> D[Pool di Agenti 2]
    B --> E[Pool di Agenti 3]
    C --> F[Aggregazione dei Risultati]
    D --> F
    E --> F
    F --> G[Output Finale]
    
    H[API Modelli GitHub] --> C
    H --> D
    H --> E
```
  
**Benefici Chiave:**  
- **⚡ Prestazioni**: Incremento significativo della velocità tramite esecuzione parallela  
- **📈 Scalabilità**: Gestisci carichi maggiori senza aumenti proporzionali del tempo  
- **🔄 Efficienza**: Migliore utilizzo delle risorse computazionali disponibili  
- **🎯 Throughput**: Elaborare più lavoro nello stesso tempo

## 🎨 **Modelli di Progettazione di Flussi di Lavoro Concorrenti**

### 🔍 **Pipeline di Ricerca e Analisi**  
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```
  
### 📊 **Flusso di Lavoro di Elaborazione Dati**  
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```
  
### 🎭 **Pipeline di Creazione Contenuti**  
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```
  
### 🔄 **Elaborazione Multi-Fase**  
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```
  
## 🏢 **Vantaggi di Prestazione Aziendali**

### ⚡ **Ottimizzazione del Throughput**
- **Esecuzione Parallela**: Più agenti che lavorano simultaneamente
- **Utilizzo Risorse**: Massima efficienza della capacità modello AI disponibile
- **Riduzione Tempo**: Diminuzione significativa del tempo di elaborazione totale
- **Architettura Scalabile**: Aggiungi facilmente più agenti concorrenti secondo necessità

### 🛡️ **Affidabilità e Resilienza**
- **Tolleranza ai Guasti**: Fallimenti di singoli agenti non interrompono l’intero flusso
- **Isolamento degli Errori**: Problemi in un ramo concorrente non impattano gli altri
- **Degrado Graduale**: Il sistema continua a operare anche con capacità agente ridotta
- **Meccanismi di Recupero**: Ritentativi automatici e gestione errori per operazioni fallite

### 📊 **Monitoraggio e Osservabilità**
- **Tracciamento dell’Esecuzione Concorrente**: Monitora il progresso di tutte le operazioni parallele
- **Metriche di Prestazione**: Misura l’aumento di velocità e i guadagni di efficienza
- **Analisi dell’Utilizzo Risorse**: Ottimizza l’allocazione degli agenti concorrenti
- **Individuazione Collo di Bottiglia**: Identifica e risolvi i vincoli di prestazione

Costruiamo flussi AI concorrenti ad alte prestazioni! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = await provider.create_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = await provider.create_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)

In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
